USING LANGCHAIN

### 📘 **Introduction**

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system by combining semantic search with a large language model (LLM). The goal is to accurately answer user questions by retrieving relevant information from a dataset before passing it to the LLM.

The pipeline integrates three core components:

---

#### Semantic Retrieval with Sentence Transformers  
A **bi-encoder** is used to generate embeddings for a large set of questions.  
When a user submits a query, the system compares it to the embedded dataset and retrieves the most **semantically similar questions**.

---

#### Re-ranking with a Cross-Encoder  
The initial **top-k** results are refined using a **cross-encoder**, which performs pairwise scoring between the user's query and each candidate.  
This improves accuracy by ranking based on **contextual relevance**.

---

#### Answer Generation with LangChain + LLM  
After identifying the most relevant questions and associated documents, a **prompt is dynamically constructed**.  
This prompt includes:
- the original user query  
- the retrieved documents  

It is passed to a powerful **LLM (e.g., Google Gemini)** via **LangChain**, which generates a final, **context-aware answer**.


In [28]:
# %%
!pip install -q sentence-transformers datasets==2.16.0

from sentence_transformers import SentenceTransformer, CrossEncoder, util
from datasets import load_dataset
import torch

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
CROSSENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_K = 10        # initial retrieval
FINAL_TOP_K = 3   # final top-k after re-ranking

print("Loading embedding model...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL)

print("Loading cross-encoder for re-ranking...")
cross_encoder = CrossEncoder(CROSSENCODER_MODEL)

print("Downloading dataset from Hugging Face Hub...")
dataset = load_dataset("FreedomIntelligence/RAG-Instruct", split="train", trust_remote_code=True)

questions_dataset = [item["question"] for item in dataset]
print(f"✅ {len(questions_dataset)} questions loaded.")

# Generate embeddings once
print("Generating embeddings...")
embeddings_dataset = bi_encoder.encode(questions_dataset, convert_to_tensor=True)

def retrieve_and_rerank(user_question, similarity_threshold=0.6):
    # Embed the user's question
    user_embedding = bi_encoder.encode(user_question, convert_to_tensor=True)

    # Get initial top-k candidates using cosine similarity
    similarities = util.pytorch_cos_sim(user_embedding, embeddings_dataset)[0]
    top_k = torch.topk(similarities, k=TOP_K)
    top_k_indices = top_k.indices.tolist()
    top_k_scores = top_k.values.tolist()

    # Check maximum similarity score
    max_similarity = max(top_k_scores)
    print(f"Max similarity: {max_similarity:.4f}")

    # If the max similarity is too low, return a fallback instruction
    if max_similarity < similarity_threshold:
        fallback_instruction = [
            "ERROR: no context"
        ]
        return [user_question], fallback_instruction

    # Prepare pairs for cross-encoder re-ranking
    candidate_pairs = [(user_question, questions_dataset[i]) for i in top_k_indices]

    print("Re-ranking with CrossEncoder...")
    rerank_scores = cross_encoder.predict(candidate_pairs)

    # Sort by cross-encoder score
    reranked = list(zip(top_k_indices, rerank_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    # Prepare final outputs
    final_questions = []
    final_documents = []

    for idx, score in reranked[:FINAL_TOP_K]:
        question = dataset[idx]["question"]
        documents = dataset[idx]["documents"]
        if isinstance(documents, str):
            documents = [documents]

        final_questions.append(question)
        final_documents.extend(documents)

    return [user_question], final_documents




huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading embedding model...
Loading cross-encoder for re-ranking...
✅ 40541 questions loaded.
Generating embeddings...


Batches:   0%|          | 0/1267 [00:00<?, ?it/s]

In [12]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,  # Tokenizers and models
    GPT2Tokenizer, GPT2LMHeadModel,                              # GPT-2 specific classes
    Seq2SeqTrainer, Seq2SeqTrainingArguments,                    # Seq2Seq training tools
    Trainer, TrainingArguments,                                  # General Trainer API
    TextDataset, DataCollatorWithPadding,                        # For dataset formatting
    T5ForConditionalGeneration,T5Tokenizer,
    pipeline                                                     # For easy model inference
)

from peft import LoraConfig, get_peft_model, TaskType, PeftModel 

In [14]:
flan_t5_path = "google/flan-t5-base"
t5_model = AutoModelForSeq2SeqLM.from_pretrained(flan_t5_path).to("cuda")
t5_tokenizer = AutoTokenizer.from_pretrained(flan_t5_path)

adapter_path ="/kaggle/input/flan-t5-finetuning/FLAN t5-lora-qa-deepseekk=10"
ft_model = PeftModel.from_pretrained(t5_model, adapter_path).to("cuda")

In [29]:
def generate_answer(question, documents, tokenizer, model):
    
    if (documents[0] == "ERROR: no context"):
        return "I'm sorry but I'm not able to answer this question"
    context = "\n".join(documents)
    print (context)
    # input and target for seq2seq
    input_text = f"{context}\n\nQuestion: {question}"

    # tokenize
    input_encoding = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    ).to(model.device)

    # generate prediction
    output_ids = model.generate(
        input_ids=input_encoding["input_ids"],
        attention_mask=input_encoding["attention_mask"],
        max_new_tokens=256,
        num_beams=4,
        do_sample=False
    )

    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return prediction

    
question = "Peppe bello?"
question, documents = retrieve_and_rerank(question)

print(generate_answer(question, documents, t5_tokenizer, ft_model))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Max similarity: 0.3741
I'm sorry but I'm not able to answer this question
